# 🌟 Pixel-Precise Solar Filament Instance Segmentation
## IEEE Big Data Cup 2026 — Kaggle Competition

---

### Team
| Member | Institution |
|---|---|
| Daichi Sasaki | Hiroshima University, Japan |
| Saif Nasr | Chonnam National University, South Korea |
| Yujeong Lee | Kyungpook National University, South Korea |
| Seungyoon Im | Kyungpook National University, South Korea |
| Seo-hoo Han | Pusan National University, South Korea |

---

### Pipeline Overview
- **Models:** U-Net ResNet-34 × 5 folds + EfficientNet-B4 × 5 folds + RF-DETR-Seg-Large × 5 folds
- **Training:** 5-fold cross-validation · mean soft labels · EMA · cosine schedule
- **Inference:** D4 TTA · overlapping tiles · Hann window blending
- **Selector:** BlendSelector (hurdle + LambdaRank) · 40 features · DP antichain
- **Metric:** Panoptic Quality (PQ = SQ × RQ)
- **OOF PQ:** 0.4665 · **LB Score:** 0.41

---

### Notebook Structure
1. Environment Setup
2. Data Loading & Fold Splits
3. Model Definition
4. Inference Functions
5. U-Net Training *(skippable — pretrained weights provided)*
6. RF-DETR Training *(skippable — pretrained weights provided)*
7. Load Pretrained Weights
8. OOF Probability Maps *(skippable — prebuilt cache provided)*
9. Component Tree & Feature Extraction
10. OOF Cache *(skippable — prebuilt cache provided)*
11. Load OOF Cache
12. Selector
13. Test Probability Maps
14. RF-DETR Inference
15. Generate Submission

## 1. Environment Setup

In [ ]:
# ── Imports ───────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# ── RF-DETR ───────────────────────────────────────────────────────
try:
    import rfdetr
    print("rfdetr ready")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "rfdetr==1.11.2"], check=True)
    import rfdetr
    print("rfdetr installed")

# ── Device ────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

# ── Paths ─────────────────────────────────────────────────────────
# Auto-detects repository root from notebook location
# Override WORK_DIR manually if needed
WORK_DIR = str(Path().resolve().parent)  # assumes pipeline.ipynb is in notebooks/
# WORK_DIR = "/path/to/repo"  # uncomment and set manually if needed
DATA_DIR         = os.path.join(WORK_DIR, "data", "MAGFiLO_1.0_Kaggle_2026")
ANNOTATIONS_PATH = os.path.join(DATA_DIR, "train", "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
SPLITS_PATH      = os.path.join(DATA_DIR, "splits", "folds_v1.csv")
print("Ready.")

/home/seif/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rfdetr ready
Device: cuda
GPU:  NVIDIA RTX A6000
VRAM: 51.0 GB
Ready.


## 2. Data Loading & Fold Splits

707 H-alpha solar images, up to 3 annotators per image. Splits stratified by annotator count and observation site.

In [2]:
import json

# ── Load annotations ──────────────────────────────────────────────
# COCO-format JSON with multi-annotator records
# Each unique image may have 1–3 annotator entries
with open(ANNOTATIONS_PATH, "r") as f:
    data = json.load(f)

print(f"Images:        {len(data['images'])}")
print(f"Annotations:   {len(data['annotations'])}")

# ── Build lookup dictionaries ─────────────────────────────────────
# image_id_to_info: fast lookup of image metadata by ID
# stem_to_ids: maps each unique filename stem to all annotator record IDs
image_id_to_info = {img["id"]: img for img in data["images"]}

stem_to_ids = {}
for img in data["images"]:
    stem = Path(img["file_name"]).stem
    stem_to_ids.setdefault(stem, []).append(img["id"])

print(f"Unique images: {len(stem_to_ids)}")

Images:        1154
Annotations:   8199
Unique images: 707


In [3]:
# ══════════════════════════════════════════════════════════════════
# REFERENCE ONLY — Do not run this cell
# Shows how fold splits were generated for reproducibility
# Pre-computed splits are loaded in the next cell
# ══════════════════════════════════════════════════════════════════

from sklearn.model_selection import StratifiedKFold

def generate_fold_splits(data, n_folds=5, random_state=42):
    stem_info = {}
    for img in data["images"]:
        stem = Path(img["file_name"]).stem
        if stem not in stem_info:
            stem_info[stem] = {"n_ann": 0, "site": stem[-1]}
        stem_info[stem]["n_ann"] += 1
    stems  = list(stem_info.keys())
    labels = [
        f"{min(stem_info[s]['n_ann'], 3)}_{stem_info[s]['site']}"
        for s in stems
    ]
    skf  = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    rows = []
    for fold_idx, (_, val_idx) in enumerate(skf.split(stems, labels)):
        for i in val_idx:
            rows.append({"image_stem": stems[i], "fold": fold_idx})
    return pd.DataFrame(rows)

In [4]:
# ── Load pre-computed splits ──────────────────────────────────────
fold_splits_df = pd.read_csv(SPLITS_PATH)
N_FOLDS = 5

fold_splits = []
for fold_idx in range(N_FOLDS):
    val_df   = fold_splits_df[fold_splits_df["fold"] == fold_idx]
    train_df = fold_splits_df[fold_splits_df["fold"] != fold_idx]
    fold_splits.append({
        "train": train_df["image_stem"].tolist(),
        "val":   val_df["image_stem"].tolist(),
    })
    print(f"Fold {fold_idx+1}: {len(train_df)} train | {len(val_df)} val")

Fold 1: 565 train | 142 val
Fold 2: 565 train | 142 val
Fold 3: 566 train | 141 val
Fold 4: 566 train | 141 val
Fold 5: 566 train | 141 val


## 3. Model Definition

U-Net segmentation model with interchangeable encoders.
ResNet-34 and EfficientNet-B4 encoders are used in the ensemble,
both pretrained on ImageNet.

In [5]:
import segmentation_models_pytorch as smp

# ── Model builder ─────────────────────────────────────────────────
# U-Net with configurable encoder — same architecture for R34 and B4
# in_channels=1: grayscale H-alpha images
# classes=1: binary filament probability map

def build_model(encoder_name, encoder_weights="imagenet"):
    model = smp.Unet(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=1,
        classes=1,
    )
    return model


def unwrap_model(model):
    """Unwrap DataParallel if present."""
    return model.module if isinstance(model, nn.DataParallel) else model


print("Model definition ready.")
print(f"  R34: {sum(p.numel() for p in build_model('resnet34').parameters()):,} parameters")
print(f"  B4:  {sum(p.numel() for p in build_model('efficientnet-b4').parameters()):,} parameters")

Model definition ready.
  R34: 24,430,097 parameters
  B4:  20,224,825 parameters


## 4. Inference Functions

Full-resolution inference using overlapping tiles with Hann window blending.
D4 TTA averages predictions across all 8 symmetries of the square (4 rotations × 2 flips),
improving robustness without additional training.

In [6]:
import cv2
from PIL import Image

TILE_SIZE            = 512
STRIDE               = 256   # 50% overlap — reduces tile boundary artifacts
INFERENCE_BATCH_SIZE = 32

# ── Hann window ───────────────────────────────────────────────────
# Smooth blending weight — downweights tile edges where predictions
# are less reliable due to limited context
def _hann_window(tile_size):
    w = np.maximum(np.hanning(tile_size), 1e-3)
    w = np.outer(w, w).astype(np.float32)
    return w / w.max()


def _starts(length, tile, stride):
    """Tile start positions covering the full length."""
    if length <= tile:
        return [0]
    s = list(range(0, length - tile + 1, stride))
    if s[-1] != length - tile:
        s.append(length - tile)
    return s


# ── Single model inference ────────────────────────────────────────
@torch.no_grad()
def predict_full_image(model, image_path, device,
                       tile_size=TILE_SIZE, stride=STRIDE,
                       batch_size=INFERENCE_BATCH_SIZE, amp=True):
    """Full-resolution inference with overlapping tiles and Hann blending."""
    image  = np.array(Image.open(image_path).convert("L"))
    h, w   = image.shape
    win    = _hann_window(tile_size)
    prob   = np.zeros((h, w), dtype=np.float32)
    weight = np.zeros((h, w), dtype=np.float32)
    coords = [(y, x)
              for y in _starts(h, tile_size, stride)
              for x in _starts(w, tile_size, stride)]
    model.eval()
    for i in range(0, len(coords), batch_size):
        batch = coords[i:i+batch_size]
        tiles = np.stack([image[y:y+tile_size, x:x+tile_size] for y, x in batch])
        x_t   = torch.from_numpy(tiles).float().div_(255).unsqueeze(1)
        x_t   = x_t.to(device, non_blocking=True).contiguous(
            memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type,
                            enabled=amp and device.type == "cuda"):
            p = torch.sigmoid(model(x_t).float()).squeeze(1).cpu().numpy()
        for (y, x), tile_p in zip(batch, p):
            prob  [y:y+tile_size, x:x+tile_size] += tile_p * win
            weight[y:y+tile_size, x:x+tile_size] += win
    return prob / np.maximum(weight, 1e-6)


# ── Array-based inference (used by TTA) ──────────────────────────
def _predict_array(model, image, device,
                   tile_size=TILE_SIZE, stride=STRIDE,
                   batch_size=INFERENCE_BATCH_SIZE, amp=True):
    """Same as predict_full_image but accepts numpy array directly."""
    h, w   = image.shape
    win    = _hann_window(tile_size)
    prob   = np.zeros((h, w), dtype=np.float32)
    weight = np.zeros((h, w), dtype=np.float32)
    coords = [(y, x)
              for y in _starts(h, tile_size, stride)
              for x in _starts(w, tile_size, stride)]
    model.eval()
    for i in range(0, len(coords), batch_size):
        batch = coords[i:i+batch_size]
        tiles = np.stack([image[y:y+tile_size, x:x+tile_size] for y, x in batch])
        x_t   = torch.from_numpy(tiles).float().div_(255).unsqueeze(1)
        x_t   = x_t.to(device, non_blocking=True).contiguous(
            memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type,
                            enabled=amp and device.type == "cuda"):
            p = torch.sigmoid(model(x_t).float()).squeeze(1).cpu().numpy()
        for (y, x), tile_p in zip(batch, p):
            prob  [y:y+tile_size, x:x+tile_size] += tile_p * win
            weight[y:y+tile_size, x:x+tile_size] += win
    return prob / np.maximum(weight, 1e-6)


# ── D4 TTA ────────────────────────────────────────────────────────
# Averages predictions across 8 dihedral transforms
# Each transform is inverted before averaging so all maps
# are in the original image coordinate system
@torch.no_grad()
def predict_full_image_d4tta(model, image_path, device):
    """D4 TTA — average of 8 dihedral transforms."""
    image = np.array(Image.open(image_path).convert("L"))
    maps  = []
    for k in range(4):
        for flip in (False, True):
            t = np.rot90(image, k)
            t = np.ascontiguousarray(t[:, ::-1] if flip else t)
            p = _predict_array(model, t, device)
            if flip:
                p = p[:, ::-1]
            maps.append(np.rot90(p, -k))
    return np.mean(maps, axis=0).astype(np.float32)

def save_prob(path, prob):
    """Save probability map as float16 .npz."""
    np.savez_compressed(path, prob=prob.astype(np.float16))

def load_prob(path):
    """Load probability map from .npz."""
    with np.load(path) as f:
        return f["prob"].astype(np.float32)

print("Inference functions ready.")

Inference functions ready.


## 5. Training

*(Skip this section if using pretrained weights — see Section 6)*

Two U-Net models are trained independently using 5-fold cross-validation:
- **ResNet-34** encoder 
- **EfficientNet-B4** encoder 

Pretrained weights are provided in Section 7.

### 5.1 Hyperparameters & Loss

In [7]:
# ══════════════════════════════════════════════════════════════════
# OPTIONAL — Skip if using pretrained weights (Section 6)
# ══════════════════════════════════════════════════════════════════

import copy
import time
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

# ── Hyperparameters ───────────────────────────────────────────────
NUM_EPOCHS       = 20
BATCH_SIZE       = 16
NEG_TO_POS_RATIO = 3       # negative:positive tile ratio
MAX_TRAIN_TILES  = 12000   # fixed tile list size per fold
EMA_DECAY        = 0.999   # exponential moving average decay
EVAL_EVERY       = 2       # validate every N epochs
WARMUP_FRAC      = 0.03    # fraction of steps for LR warmup
MIN_LR_FRAC      = 0.02    # cosine schedule minimum LR fraction

# ── Loss ──────────────────────────────────────────────────────────
# BCE (0.98) + Dice (0.02)
# Small Dice weight avoids instability early in training
# Dice is batch-pooled (dims=0,1,2,3) for stable gradients

class DiceLoss(nn.Module):
    def __init__(self, smooth=0.1):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        dims  = (0, 1, 2, 3)
        inter = (probs * targets).sum(dims)
        union = probs.sum(dims) + targets.sum(dims)
        return 1 - (2 * inter + self.smooth) / (union + self.smooth)


class CompositeLoss(nn.Module):
    def __init__(self, w_bce=0.98, w_dice=0.02):
        super().__init__()
        self.bce    = nn.BCEWithLogitsLoss()
        self.dice   = DiceLoss(smooth=0.1)
        self.w_bce  = w_bce
        self.w_dice = w_dice

    def forward(self, logits, targets):
        bce  = self.bce(logits, targets)
        dice = self.dice(logits, targets)
        loss = self.w_bce * bce + self.w_dice * dice
        return loss, (bce, dice)


def build_loss():
    return CompositeLoss(w_bce=0.98, w_dice=0.02)


# ── Metrics ───────────────────────────────────────────────────────
# Tile-level metrics for training monitoring only
# Checkpoint selection uses full-image PQ (see Cell 4c)

def dice_coefficient(logits, targets, threshold=0.5, smooth=1e-6):
    preds = (torch.sigmoid(logits) > threshold).float()
    inter = (preds * targets).sum()
    return ((2 * inter + smooth) / (preds.sum() + targets.sum() + smooth)).item()


def iou_score(logits, targets, threshold=0.5, smooth=1e-6):
    preds = (torch.sigmoid(logits) > threshold).float()
    inter = (preds * targets).sum()
    union = preds.sum() + targets.sum() - inter
    return ((inter + smooth) / (union + smooth)).item()

print("Hyperparameters and loss ready.")

Hyperparameters and loss ready.


### 5.2 Dataset & Tile Builder

In [8]:
# ══════════════════════════════════════════════════════════════════
# OPTIONAL — Skip if using pretrained weights (Section 6)
# ══════════════════════════════════════════════════════════════════

from torch.utils.data import Dataset, DataLoader

# ── Dataset ───────────────────────────────────────────────────────
# Mean soft labels: target = average mask over all annotators
# Per-sample reproducible RNG ensures consistent augmentation
# D4 augmentation: 4 rotations × 2 flips + brightness jitter

class FilamentTileDataset(Dataset):
    def __init__(self, tiles, augment=False):
        self.tiles   = tiles
        self.augment = augment

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):
        tile  = self.tiles[idx]
        image = tile["image"].astype(np.float32) / 255.0
        mask  = tile["mask"].astype(np.float32)

        worker = torch.utils.data.get_worker_info()
        wseed  = worker.seed if worker else 0
        rng    = np.random.default_rng([wseed, idx])

        if self.augment:
            # D4: random rotation + horizontal flip
            k     = rng.integers(4)
            image = np.rot90(image, k).copy()
            mask  = np.rot90(mask,  k).copy()
            if rng.random() < 0.5:
                image = np.fliplr(image).copy()
                mask  = np.fliplr(mask).copy()
            # Brightness jitter
            gain   = rng.uniform(0.9, 1.1)
            offset = rng.uniform(-0.03, 0.03)
            image  = np.clip(image * gain + offset, 0, 1)

        image = torch.from_numpy(image).unsqueeze(0)
        mask  = torch.from_numpy(mask).unsqueeze(0)
        return image, mask


# ── Tile builder ──────────────────────────────────────────────────
# Extracts 512×512 tiles from 2048×2048 images
# Builds mean soft-label mask over all annotators per image
# Separates tiles into positive (has filament) and negative

def build_fold_tiles(fold_idx, fold_splits, data, images_dir, tile_size=TILE_SIZE):
    val_stems   = set(fold_splits[fold_idx]["val"])
    train_stems = set(fold_splits[fold_idx]["train"])

    stem_to_ids_local = {}
    for img in data["images"]:
        s = Path(img["file_name"]).stem
        stem_to_ids_local.setdefault(s, []).append(img["id"])

    pos_train, neg_train, val_tiles = [], [], []

    for img_info in tqdm(data["images"], desc=f"Fold {fold_idx+1} tiles"):
        stem       = Path(img_info["file_name"]).stem
        image_path = os.path.join(images_dir, img_info["file_name"])
        image      = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            continue
        h, w = image.shape

        # Mean soft-label mask over all annotators
        all_ids  = stem_to_ids_local.get(stem, [img_info["id"]])
        mask_sum = np.zeros((h, w), dtype=np.float32)
        for aid in all_ids:
            ann_mask = np.zeros((h, w), dtype=np.float32)
            for ann in data["annotations"]:
                if ann["image_id"] != aid:
                    continue
                for poly in ann["segmentation"]:
                    pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
                    cv2.fillPoly(ann_mask, [pts], 1.0)
            mask_sum += ann_mask
        mean_mask = mask_sum / len(all_ids)

        is_val   = stem in val_stems
        is_train = stem in train_stems

        for y0 in range(0, h - tile_size + 1, tile_size):
            for x0 in range(0, w - tile_size + 1, tile_size):
                img_tile  = image    [y0:y0+tile_size, x0:x0+tile_size]
                mask_tile = mean_mask[y0:y0+tile_size, x0:x0+tile_size]
                has_fil   = mask_tile.max() > 0.05
                tile      = {"image": img_tile, "mask": mask_tile,
                             "has_filament": has_fil, "stem": stem}
                if is_val:
                    val_tiles.append(tile)
                elif is_train:
                    (pos_train if has_fil else neg_train).append(tile)

    return pos_train, neg_train, val_tiles


def build_fold_dataloaders(all_tiles, batch_size=BATCH_SIZE):
    pos_train, neg_train, val_tiles = all_tiles
    num_workers = 4
    pin_memory  = torch.cuda.is_available()

    val_dataset = FilamentTileDataset(val_tiles, augment=False)
    val_loader  = DataLoader(val_dataset, batch_size=batch_size * 2,
                             shuffle=False, num_workers=num_workers,
                             pin_memory=pin_memory, persistent_workers=False)
    return pos_train, neg_train, val_loader, batch_size, num_workers, pin_memory

print("Dataset and tile builder ready.")

Dataset and tile builder ready.


### 5.3 Checkpoint & Validation

In [9]:
# ══════════════════════════════════════════════════════════════════
# OPTIONAL — Skip if using pretrained weights (Section 6)
# ══════════════════════════════════════════════════════════════════

from scipy import ndimage as ndi

# ── Fast PQ using bbox intersection ──────────────────────────────
# Bbox-clipped IoU avoids full 2048×2048 mask comparisons
def compute_iou_bbox(cand, gt_mask):
    y0, x0, y1, x1, crop = cand
    gt_crop = gt_mask[y0:y1, x0:x1]
    inter   = int((crop & gt_crop).sum())
    if inter == 0:
        return 0.0
    union = int(crop.sum()) + int(gt_mask.sum()) - inter
    return inter / union if union > 0 else 0.0


def compute_pq_fast_bbox(pred_cands, gt_masks, iou_threshold=0.5):
    if len(gt_masks) == 0 and len(pred_cands) == 0:
        return 1.0, 1.0, 1.0
    if len(gt_masks) == 0:
        return 0.0, 0.0, 0.0
    if len(pred_cands) == 0:
        return 0.0, 0.0, 0.0
    iou_matrix = np.zeros((len(pred_cands), len(gt_masks)), dtype=np.float32)
    for i, cand in enumerate(pred_cands):
        for j, gt_mask in enumerate(gt_masks):
            iou_matrix[i, j] = compute_iou_bbox(cand, gt_mask)
    matched_gt = set()
    tp_iou_sum = 0.0
    for i in range(len(pred_cands)):
        best_j = iou_matrix[i].argmax()
        if iou_matrix[i, best_j] >= iou_threshold and best_j not in matched_gt:
            tp_iou_sum += iou_matrix[i, best_j]
            matched_gt.add(best_j)
    tp = len(matched_gt)
    fp = len(pred_cands) - tp
    fn = len(gt_masks)   - tp
    sq = tp_iou_sum / tp if tp > 0 else 0.0
    rq = tp / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) > 0 else 0.0
    return sq * rq, sq, rq


# ── Checkpoint ────────────────────────────────────────────────────
# Saves EMA model weights — EMA is used for all inference
def save_checkpoint(path, model, epoch, fold_idx, val_pq=None, ema_model=None):
    m = unwrap_model(ema_model if ema_model is not None else model)
    torch.save({
        "model":    m.state_dict(),
        "epoch":    int(epoch),
        "fold_idx": fold_idx,
        "val_pq":   float(val_pq) if val_pq is not None else -1.0,
        "ema":      ema_model is not None,
    }, path)


# ── Tile-level validation ─────────────────────────────────────────
# Used for monitoring only — not for checkpoint selection
def validate_one_epoch(model, loader, criterion, device, epoch):
    model.eval()
    running_loss = running_dice = running_iou = samples_seen = 0.0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs  = imgs.to(device, non_blocking=True).contiguous(
                memory_format=torch.channels_last)
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=device.type == "cuda"):
                logits  = model(imgs)
                loss, _ = criterion(logits, masks)
            bs            = imgs.size(0)
            samples_seen += bs
            running_loss += loss.item() * bs
            running_dice += dice_coefficient(logits, masks) * bs
            running_iou  += iou_score(logits, masks) * bs
    n = max(samples_seen, 1)
    return running_loss / n, running_dice / n, running_iou / n


# ── Full-image PQ validation ──────────────────────────────────────
# Checkpoint selection uses full-image PQ — directly optimizes
# the competition metric. Five thresholds are tried per image;
# the best PQ across thresholds is used to avoid penalizing
# models whose probability scale shifts during training.
def evaluate_val_pq(ema_model, val_stems, stem_to_path_local, data, device,
                    thresholds=(0.3, 0.4, 0.5, 0.6, 0.7),
                    closing=2, connectivity=8,
                    min_pixels=512, max_instances=80):

    stem_to_ids_local = {}
    for img in data["images"]:
        s = Path(img["file_name"]).stem
        stem_to_ids_local.setdefault(s, []).append(img["id"])

    ema_model.eval()
    best_pq_per_image = []

    with torch.no_grad():
        for stem in val_stems:
            image_path = stem_to_path_local[stem]
            image      = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            prob       = predict_full_image(ema_model, image_path, device)

            H, W     = image.shape
            gt_masks = []
            for aid in stem_to_ids_local.get(stem, []):
                anns     = [a for a in data["annotations"] if a["image_id"] == aid]
                ann_mask = np.zeros((H, W), dtype=np.uint8)
                for ann in anns:
                    for poly in ann["segmentation"]:
                        pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
                        cv2.fillPoly(ann_mask, [pts], 1)
                n2, lab2, _, _ = cv2.connectedComponentsWithStats(
                    ann_mask, connectivity=8)
                for k in range(1, n2):
                    gt_masks.append((lab2 == k))

            best_pq = 0.0
            for tau in thresholds:
                mask = ndi.binary_closing(
                    (prob > tau), iterations=closing).astype(np.uint8)
                n, labels, stats, _ = cv2.connectedComponentsWithStats(
                    mask, connectivity=connectivity)
                pred_cands = []
                for k in range(1, n):
                    area = stats[k, cv2.CC_STAT_AREA]
                    if area < min_pixels:
                        continue
                    x, y   = stats[k, cv2.CC_STAT_LEFT], stats[k, cv2.CC_STAT_TOP]
                    ww, hh = stats[k, cv2.CC_STAT_WIDTH], stats[k, cv2.CC_STAT_HEIGHT]
                    crop   = (labels[y:y+hh, x:x+ww] == k)
                    pred_cands.append((y, x, y+hh, x+ww, crop))
                pred_cands = sorted(pred_cands,
                                    key=lambda c: c[4].sum(),
                                    reverse=True)[:max_instances]
                pq, _, _   = compute_pq_fast_bbox(pred_cands, gt_masks)
                best_pq    = max(best_pq, pq)

            best_pq_per_image.append(best_pq)

    return float(np.mean(best_pq_per_image))

print("Checkpoint and validation functions ready.")

Checkpoint and validation functions ready.


### 5.4 Training Loop

In [ ]:
# ══════════════════════════════════════════════════════════════════
# OPTIONAL — Skip if using pretrained weights (Section 6)
# ══════════════════════════════════════════════════════════════════

TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, "train", "train_images")
LOG_EVERY               = 10
HEARTBEAT_EVERY_SECONDS = 60

for ENCODER_NAME in ["resnet34", "efficientnet-b4"]:
    suffix         = "r34" if ENCODER_NAME == "resnet34" else "b4"
    training_start = time.time()

    for fold_idx in range(N_FOLDS):
        print(f"\n{'='*50}")
        print(f"Training {ENCODER_NAME} — Fold {fold_idx+1}/{N_FOLDS}")
        print(f"{'='*50}\n")

        all_tiles = build_fold_tiles(fold_idx, fold_splits, data, TRAIN_IMAGES_DIR)
        pos_train, neg_train, val_loader, batch_size, num_workers, PIN_MEMORY = \
            build_fold_dataloaders(all_tiles)

        # Fixed tile list — computed once per fold, shuffled each epoch
        # neg_to_pos_ratio=3 reduces false positives
        rng_init = np.random.default_rng(42 + fold_idx)
        n_neg    = min(len(neg_train), NEG_TO_POS_RATIO * len(pos_train))
        neg_idx  = rng_init.choice(len(neg_train), size=n_neg, replace=False)
        fixed_tiles = pos_train + [neg_train[i] for i in neg_idx]
        if len(fixed_tiles) > MAX_TRAIN_TILES:
            rng_init.shuffle(fixed_tiles)
            fixed_tiles = fixed_tiles[:MAX_TRAIN_TILES]
        print(f"Tiles: {len(fixed_tiles)} "
              f"(pos={sum(t['has_filament'] for t in fixed_tiles)}, "
              f"neg={sum(not t['has_filament'] for t in fixed_tiles)})")

        # Model
        model = build_model(ENCODER_NAME).to(device)
        if torch.cuda.is_available():
            model = model.to(memory_format=torch.channels_last)

        # EMA — smooths weight updates, used for all inference
        ema_model = copy.deepcopy(unwrap_model(model)).to(device)
        ema_model.eval()
        for p in ema_model.parameters():
            p.requires_grad_(False)
        ema_updates = 0

        def update_ema(ema_model, model, ema_updates):
            ema_updates += 1
            d   = min(EMA_DECAY, (1 + ema_updates) / (10 + ema_updates))
            src = unwrap_model(model).state_dict()
            for k, v in ema_model.state_dict().items():
                if v.dtype.is_floating_point:
                    v.mul_(d).add_(src[k].detach(), alpha=1 - d)
                else:
                    v.copy_(src[k])
            return ema_updates

        # Loss + optimizer + cosine scheduler with warmup
        criterion    = build_loss()
        n_steps      = NUM_EPOCHS * (MAX_TRAIN_TILES // BATCH_SIZE)
        warmup_steps = int(WARMUP_FRAC * n_steps)
        optimizer    = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

        def lr_lambda(step):
            if step < warmup_steps:
                return (step + 1) / warmup_steps
            t = min((step - warmup_steps) / max(1, n_steps - warmup_steps), 1.0)
            return MIN_LR_FRAC + (1 - MIN_LR_FRAC) * 0.5 * (1 + np.cos(np.pi * t))

        scheduler = LambdaLR(optimizer, lr_lambda)
        scaler    = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

        # Paths — weights/r34/fold0/last.pth
        fold_dir     = os.path.join(WEIGHTS_DIR, suffix, f"fold{fold_idx}")
        os.makedirs(fold_dir, exist_ok=True)
        LAST_PATH    = os.path.join(fold_dir, "last.pth")
        HISTORY_PATH = os.path.join(fold_dir, "history.csv")

        history         = {"epoch": [], "train_loss": [], "val_loss": [],
                           "val_dice": [], "val_pq": [], "minutes": [], "lr": []}
        best_val_metric = -1.0

        # Val stems + paths for PQ evaluation
        val_stems          = fold_splits[fold_idx]["val"]
        stem_to_path_local = {
            Path(img["file_name"]).stem: os.path.join(TRAIN_IMAGES_DIR, img["file_name"])
            for img in data["images"]
        }

        for epoch in range(1, NUM_EPOCHS + 1):
            epoch_start = time.time()

            # Shuffle fixed tiles each epoch with reproducible seed
            rng_epoch   = np.random.default_rng(epoch + fold_idx * 100)
            epoch_tiles = fixed_tiles.copy()
            rng_epoch.shuffle(epoch_tiles)

            train_dataset = FilamentTileDataset(epoch_tiles, augment=True)
            train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                       shuffle=True, num_workers=num_workers,
                                       pin_memory=PIN_MEMORY,
                                       persistent_workers=False, drop_last=True)

            # Train one epoch
            model.train()
            running_loss = samples_seen = 0.0
            last_hb      = time.time()
            n_batches    = len(train_loader)

            pbar = tqdm(train_loader, desc=f"E{epoch:02d} train",
                        leave=True, total=n_batches, mininterval=5)

            for batch_idx, (imgs, masks) in enumerate(pbar, 1):
                imgs  = imgs.to(device, non_blocking=True).contiguous(
                    memory_format=torch.channels_last)
                masks = masks.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                    logits      = model(imgs)
                    loss, parts = criterion(logits, masks)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                ema_updates = update_ema(ema_model, model, ema_updates)

                bs            = imgs.size(0)
                samples_seen += bs
                running_loss += loss.item() * bs
                avg_loss      = running_loss / samples_seen
                pbar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{avg_loss:.4f}")

                now = time.time()
                if (batch_idx == 1 or batch_idx % LOG_EVERY == 0 or
                        batch_idx == n_batches or
                        (now - last_hb) >= HEARTBEAT_EVERY_SECONDS):
                    elapsed = now - epoch_start
                    eta_min = (n_batches - batch_idx) / \
                              max(batch_idx / max(elapsed, 1e-6), 1e-6) / 60
                    print(f"E{epoch:02d} [{batch_idx:04d}/{n_batches}] "
                          f"loss={loss.item():.4f} avg={avg_loss:.4f} "
                          f"lr={optimizer.param_groups[0]['lr']:.2e} "
                          f"eta={eta_min:.1f}m", flush=True)
                    last_hb = now

            train_loss = running_loss / max(samples_seen, 1)

            # Validate every EVAL_EVERY epochs using full-image PQ
            if epoch % EVAL_EVERY == 0 or epoch == NUM_EPOCHS:
                val_loss, val_dice, _ = validate_one_epoch(
                    ema_model, val_loader, criterion, device, epoch)
                val_pq        = evaluate_val_pq(
                    ema_model, val_stems, stem_to_path_local, data, device)
                epoch_minutes = (time.time() - epoch_start) / 60
                current_lr    = optimizer.param_groups[0]["lr"]

                history["epoch"].append(epoch)
                history["train_loss"].append(train_loss)
                history["val_loss"].append(val_loss)
                history["val_dice"].append(val_dice)
                history["val_pq"].append(val_pq)
                history["minutes"].append(epoch_minutes)
                history["lr"].append(current_lr)
                pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

                print(f"Fold {fold_idx+1} E{epoch:02d}/{NUM_EPOCHS} | "
                      f"train={train_loss:.4f} | dice={val_dice:.4f} | "
                      f"pq={val_pq:.4f} | lr={current_lr:.2e} | "
                      f"time={epoch_minutes:.1f}m", flush=True)

                if val_pq > best_val_metric:
                    best_val_metric = val_pq

        # Save last epoch — cosine schedule ensures last ≈ best
        save_checkpoint(LAST_PATH, model, NUM_EPOCHS, fold_idx,
                        val_pq=val_pq, ema_model=ema_model)
        print(f"Fold {fold_idx+1} done — best val_pq={best_val_metric:.4f}", flush=True)

    print(f"\n{ENCODER_NAME} done in {(time.time()-training_start)/60:.1f} min")

## 6. RF-DETR Training *(skippable — pretrained weights provided)*

RF-DETR-Seg-Large detects individual filament instances directly from the image.
Its detections are used as 4 extra features for the LightGBM selector (Section 12).
Trained on the same 5-fold split as the U-Net ensemble.

**Skip if** `weights/rfdetr/fold{0-4}/checkpoint_best_ema.pth` is available.

### 6.1 Prepare COCO Dataset

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

RFDETR_WEIGHTS_DIR = os.path.join(WORK_DIR, "weights", "rfdetr")
RFDETR_DATA_DIR    = os.path.join(WORK_DIR, "data", "rfdetr")
VAL_FRAC           = 0.10
RFDETR_SEED        = 0
COCO_CATEGORIES    = [{"id": 0, "name": "filament", "supercategory": "none"}]

def make_coco_fold(fold_idx, fold_splits, data, images_dir, out_dir, val_frac=VAL_FRAC, seed=RFDETR_SEED):
    """Convert MAGFiLO annotations to COCO format for one fold."""
    os.makedirs(out_dir, exist_ok=True)

    # Held-out stems (fold k) — never used for training or validation
    held_out = set(fold_splits[fold_idx]["val"])

    # All other stems → split into train/valid
    other_stems = [Path(img["file_name"]).stem for img in data["images"]
                   if Path(img["file_name"]).stem not in held_out]
    other_stems = sorted(set(other_stems))

    # Stratify by n_annotators × site
    stem_to_ids_local = {}
    for img in data["images"]:
        s = Path(img["file_name"]).stem
        stem_to_ids_local.setdefault(s, []).append(img["id"])

    n_ann   = np.array([len(stem_to_ids_local[s]) for s in other_stems])
    sites   = np.array([s[-1] for s in other_stems])
    strat   = [f"{n}_{si}" for n, si in zip(n_ann, sites)]

    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    tr_idx, va_idx = next(sss.split(np.zeros(len(other_stems)), strat))
    train_stems = {other_stems[i] for i in tr_idx}
    valid_stems = {other_stems[i] for i in va_idx}

    # Build COCO dicts
    ann_by_id = {}
    for ann in data["annotations"]:
        ann_by_id.setdefault(ann["image_id"], []).append(ann)

    def build_coco(stems):
        images, annotations = [], []
        entry_id = 1
        ann_id   = 1
        links    = []
        for img in data["images"]:
            stem = Path(img["file_name"]).stem
            if stem not in stems:
                continue
            images.append({
                "id":        entry_id,
                "file_name": f"{entry_id}.jpeg",
                "width":     img["width"],
                "height":    img["height"],
            })
            links.append((entry_id, img["file_name"]))
            for ann in ann_by_id.get(img["id"], []):
                annotations.append({
                    "id":          ann_id,
                    "image_id":    entry_id,
                    "category_id": 0,
                    "segmentation": ann["segmentation"],
                    "area":        float(ann["area"]),
                    "bbox":        [float(v) for v in ann["bbox"]],
                    "iscrowd":     0,
                })
                ann_id += 1
            entry_id += 1
        return {"info": {}, "licenses": [], "categories": COCO_CATEGORIES,
                "images": images, "annotations": annotations}, links

    for split, stems in (("train", train_stems), ("valid", valid_stems)):
        split_dir = os.path.join(out_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        coco, links = build_coco(stems)
        with open(os.path.join(split_dir, "_annotations.coco.json"), "w") as f:
            json.dump(coco, f)
        # Create symlinks to images
        for entry_id, file_name in links:
            src = os.path.join(images_dir, file_name)
            dst = os.path.join(split_dir, f"{entry_id}.jpeg")
            if not os.path.exists(dst):
                os.symlink(src, dst)
        print(f"  fold {fold_idx} {split}: {len(coco['images'])} records, "
              f"{len(coco['annotations'])} annotations")

    return train_stems, valid_stems, held_out


# ── Prepare COCO datasets for all folds ──────────────────────────
print("Preparing COCO datasets...")
for fold_idx in range(N_FOLDS):
    out_dir = os.path.join(RFDETR_DATA_DIR, f"fold{fold_idx}")
    if os.path.exists(os.path.join(out_dir, "train", "_annotations.coco.json")):
        print(f"  fold {fold_idx} already prepared — skipping")
        continue
    print(f"\nFold {fold_idx}:")
    make_coco_fold(fold_idx, fold_splits, data, TRAIN_IMAGES_DIR, out_dir)

print("\nCOCO datasets ready.")

### 6.2 Train RF-DETR

In [ ]:
os.environ["HF_HUB_OFFLINE"] = "1"

for fold_idx in range(N_FOLDS):
    data_dir = os.path.join(RFDETR_DATA_DIR, f"fold{fold_idx}")
    out_dir  = os.path.join(RFDETR_WEIGHTS_DIR, f"fold{fold_idx}")

    if os.path.exists(os.path.join(out_dir, "checkpoint_best_ema.pth")):
        print(f"fold {fold_idx} already trained — skipping")
        continue

    print(f"\nTraining RF-DETR fold {fold_idx}...")
    os.makedirs(out_dir, exist_ok=True)

    model = rfdetr.RFDETRSegLarge(
        pretrain_weights="rf-detr-seg-large.pt",
        resolution=864,
    )
    model.train(
        dataset_dir    = data_dir,
        output_dir     = out_dir,
        epochs         = 50,
        batch_size     = 4,
        grad_accum_steps = 4,
        lr             = 1e-4,
        lr_encoder     = 1.5e-4,
        weight_decay   = 1e-4,
        use_ema        = True,
        aug_config     = {"D4": {"p": 1.0}},
        amp_dtype      = "bf16",
        seed           = 42,
        eval_interval  = 5,
        early_stopping = True,
        early_stopping_patience = 2,
        best_model_metric = "map",
    )
    print(f"fold {fold_idx} done → {out_dir}/checkpoint_best_ema.pth")

# Verify weights
print("\nChecking RF-DETR weights...")
for fold_idx in range(N_FOLDS):
    ckpt = os.path.join(RFDETR_WEIGHTS_DIR, f"fold{fold_idx}", "checkpoint_best_ema.pth")
    print(f"  fold {fold_idx}: {'✅' if os.path.exists(ckpt) else '❌ MISSING'}")

## 7. Load Pretrained Weights

Loads the trained R34, B4 and RF-DETR models (5 folds each).
U-Net models use EMA weights from the final training epoch.
RF-DETR weights are stored as two parts and joined before loading.

In [ ]:
import segmentation_models_pytorch as smp

WEIGHTS_DIR = os.path.join(WORK_DIR, "weights")

# ── Join RF-DETR weight parts ─────────────────────────────────────
print("Joining RF-DETR weight parts...")
for fold_idx in range(N_FOLDS):
    out   = os.path.join(WEIGHTS_DIR, "rfdetr", f"fold{fold_idx}",
                         "checkpoint_best_ema.pth")
    if os.path.exists(out):
        print(f"  fold {fold_idx} already joined ✅")
        continue
    part0 = out + ".part0"
    part1 = out + ".part1"
    with open(out, "wb") as f:
        f.write(open(part0, "rb").read())
        f.write(open(part1, "rb").read())
    print(f"  fold {fold_idx} joined ✅")

# ── Load R34 models ───────────────────────────────────────────────
print("\nLoading R34 models...")
r34_models = []
for fold_idx in range(N_FOLDS):
    ckpt_path  = os.path.join(WEIGHTS_DIR, "r34", f"fold{fold_idx}", "last.pth")
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = build_model("resnet34", encoder_weights=None).to(device)
    m.load_state_dict(checkpoint["model"])
    m.eval()
    r34_models.append(m)
    print(f"  fold {fold_idx} — epoch {checkpoint['epoch']} | val_pq={checkpoint['val']['pq']:.4f}")

# ── Load B4 models ────────────────────────────────────────────────
print("\nLoading B4 models...")
b4_models = []
for fold_idx in range(N_FOLDS):
    ckpt_path  = os.path.join(WEIGHTS_DIR, "b4", f"fold{fold_idx}", "last.pth")
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = build_model("efficientnet-b4", encoder_weights=None).to(device)
    m.load_state_dict(checkpoint["model"])
    m.eval()
    b4_models.append(m)
    print(f"  fold {fold_idx} — epoch {checkpoint['epoch']} | val_pq={checkpoint['val']['pq']:.4f}")

# ── Load RF-DETR models ───────────────────────────────────────────
print("\nLoading RF-DETR models...")
rfdetr_models = []
for fold_idx in range(N_FOLDS):
    ckpt_path = os.path.join(WEIGHTS_DIR, "rfdetr", f"fold{fold_idx}",
                             "checkpoint_best_ema.pth")
    model = rfdetr.RFDETRSegLarge.from_checkpoint(ckpt_path)
    rfdetr_models.append(model)
    print(f"  fold {fold_idx} ✅")

print(f"\nLoaded {len(r34_models)} R34 + {len(b4_models)} B4 + {len(rfdetr_models)} RF-DETR models.")

Joining RF-DETR weight parts...
  fold 0 already joined ✅
  fold 1 already joined ✅
  fold 2 already joined ✅
  fold 3 already joined ✅
  fold 4 already joined ✅

Loading R34 models...
  fold 0 — epoch 20 | val_pq=0.4094
  fold 1 — epoch 20 | val_pq=0.4163
  fold 2 — epoch 20 | val_pq=0.4103
  fold 3 — epoch 20 | val_pq=0.4309
  fold 4 — epoch 20 | val_pq=0.4135

Loading B4 models...
  fold 0 — epoch 20 | val_pq=0.4113
  fold 1 — epoch 20 | val_pq=0.4281
  fold 2 — epoch 20 | val_pq=0.4225
  fold 3 — epoch 20 | val_pq=0.4251


[2026-11-12 11:19:56] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-11-12 11:19:56] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


  fold 4 — epoch 20 | val_pq=0.4169

Loading RF-DETR models...


[2026-11-12 11:19:57] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-11-12 11:19:57] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


  fold 0 ✅


[2026-11-12 11:19:58] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-11-12 11:19:58] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


  fold 1 ✅


[2026-11-12 11:20:00] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-11-12 11:20:00] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


  fold 2 ✅


[2026-11-12 11:20:01] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-11-12 11:20:01] [WARNING] rf-detr - Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


  fold 3 ✅
  fold 4 ✅

Loaded 5 R34 + 5 B4 + 5 RF-DETR models.


## 8. OOF Probability Maps *(skippable — prebuilt cache provided)*

Runs D4 TTA inference on each fold's validation images using that fold's
trained model — ensuring no image is predicted by a model that saw it during training (OOF = out-of-fold).

- **R34 × 5 folds + B4 × 5 folds** → one prob map per training image per model
- Maps are averaged (R34 + B4) to build the ensemble prob map per image
- Used in Section 10 to build the component tree cache

**Skip if `cache/tree_ens_r34_b4.pkl` is already available.**


In [ ]:
# ══════════════════════════════════════════════════════════════════
# OPTIONAL — Skip if cache/tree_ens_r34_b4.pkl is available
# ══════════════════════════════════════════════════════════════════

MAPS_DIR    = os.path.join(WORK_DIR, "maps")
ENS_DIR     = os.path.join(WORK_DIR, "maps_ens")
os.makedirs(MAPS_DIR, exist_ok=True)
os.makedirs(ENS_DIR,  exist_ok=True)

# Build stem → image info mapping
stem_to_img = {Path(img["file_name"]).stem: img for img in data["images"]}


# ── OOF inference ─────────────────────────────────────────────────
# Fold k model predicts only fold k val images → no leakage
for suffix, models in (("r34", r34_models), ("b4", b4_models)):
    for fold_idx, model in enumerate(models):
        out_dir   = os.path.join(MAPS_DIR, f"{suffix}_fold{fold_idx}")
        done_file = os.path.join(out_dir, "DONE")
        os.makedirs(out_dir, exist_ok=True)
        if os.path.exists(done_file):
            print(f"  {suffix} fold {fold_idx} already done — skipping")
            continue
        val_stems = fold_splits[fold_idx]["val"]
        print(f"\n{suffix} fold {fold_idx} — {len(val_stems)} val images")
        t0 = time.time()
        for stem in tqdm(val_stems, desc=f"{suffix}_fold{fold_idx}"):
            out_path = os.path.join(out_dir, f"{stem}.npz")
            if os.path.exists(out_path):
                continue
            img_info   = stem_to_img[stem]
            image_path = os.path.join(TRAIN_IMAGES_DIR, img_info["file_name"])
            prob       = predict_full_image_d4tta(model, image_path, device)
            save_prob(out_path, prob)
        elapsed = (time.time() - t0) / 60
        open(done_file, "w").write(f"{elapsed:.1f} min")
        print(f"  done in {elapsed:.1f} min")

# ── Ensemble OOF maps ─────────────────────────────────────────────
# Average R34 + B4 per image → one ensemble prob map per training image
print("\nBuilding ensemble OOF prob maps...")
all_stems = sorted(stem_to_img.keys())
for stem in tqdm(all_stems, desc="ensemble"):
    out_path = os.path.join(ENS_DIR, f"{stem}.npz")
    if os.path.exists(out_path):
        continue
    maps = []
    for suffix in ("r34", "b4"):
        for fold_idx in range(N_FOLDS):
            npz = os.path.join(MAPS_DIR, f"{suffix}_fold{fold_idx}", f"{stem}.npz")
            if os.path.exists(npz):
                maps.append(load_prob(npz))
    if maps:
        save_prob(out_path, np.mean(maps, axis=0))

print(f"Ensemble OOF maps saved → {ENS_DIR}")
print(f"  {len(list(Path(ENS_DIR).glob('*.npz')))} images")

## 9. Component Tree & Feature Extraction

The probability map is thresholded at 19 levels (0.05 to 0.95) to build a
component tree — a hierarchical representation of connected regions.
Each node in the tree is a candidate filament instance at a specific threshold level.

Two sets of features are extracted per node:
- **26 blob features** — shape, probability, intensity and position within the solar disk
- **10 tree features** — hierarchical structure (depth, persistence, area ratios)

4 RF-DETR detection features are added in Section 14, giving 40 features total for the selector.

In [20]:
from dataclasses import dataclass, field
from skimage.morphology import skeletonize  # ← skeletonize not thin
from scipy import ndimage as ndi

# ── Component tree ────────────────────────────────────────────────
# 19 threshold levels from 0.05 to 0.95
# Identical components on consecutive levels are merged into one node
# closing=2: fills small gaps in filaments before connected components
# connectivity=8: diagonal connections allowed for thin structures

LEVELS = [round(0.05 * k, 2) for k in range(1, 20)]

# level-0.5 candidate extraction params — reference set for image context features
CAND_PARAMS = dict(threshold=0.5, closing_iterations=2, connectivity=8,
                   min_pixels=1, max_instances=0, min_mean_prob=0.0)


@dataclass
class TreeNode:
    y0: int
    x0: int
    crop: np.ndarray  # boolean mask in tight bounding box
    area: int

    @property
    def y1(self): return self.y0 + self.crop.shape[0]
    @property
    def x1(self): return self.x0 + self.crop.shape[1]


@dataclass
class Tree:
    levels: list
    nodes:  list = field(default_factory=list)
    parent: list = field(default_factory=list)
    lo:     list = field(default_factory=list)
    hi:     list = field(default_factory=list)

    def __len__(self): return len(self.nodes)

    def children(self):
        ch = [[] for _ in self.nodes]
        for i, p in enumerate(self.parent):
            if p >= 0:
                ch[p].append(i)
        return ch

    def depth(self):
        d = np.zeros(len(self.nodes), dtype=int)
        for i, p in enumerate(self.parent):
            d[i] = 0 if p < 0 else d[p] + 1
        return d

    def at_level(self, level):
        k = self.levels.index(level)
        return [i for i in range(len(self.nodes)) if self.lo[i] <= k <= self.hi[i]]


def build_tree(prob, levels=LEVELS, closing=2, connectivity=8):
    """Build component tree from probability map."""
    tree = Tree(levels=list(levels))
    prev_labels, prev_node = None, None

    for k, t in enumerate(levels):
        mask = prob > t
        if closing:
            mask = ndi.binary_closing(mask, iterations=closing)
        n, labels, stats, _ = cv2.connectedComponentsWithStats(
            mask.astype(np.uint8), connectivity=connectivity)
        node_of = np.full(n, -1)

        for c in range(1, n):
            x, y, w, h, area = stats[c]
            crop   = labels[y:y+h, x:x+w] == c
            parent = -1
            if prev_labels is not None:
                yy, xx = np.argwhere(crop)[0]
                parent = int(prev_node[prev_labels[y+yy, x+xx]])
                if parent < 0:
                    continue
                if tree.nodes[parent].area == int(area) and tree.hi[parent] == k - 1:
                    tree.hi[parent] = k
                    node_of[c] = parent
                    continue
            node_of[c] = len(tree.nodes)
            tree.nodes.append(TreeNode(int(y), int(x), crop, int(area)))
            tree.parent.append(parent)
            tree.lo.append(k)
            tree.hi.append(k)
        prev_labels, prev_node = labels, node_of
    return tree


# ── Tree features (10) ────────────────────────────────────────────
def tree_features(tree):
    ch    = tree.children()
    depth = tree.depth()
    area  = np.array([n.area for n in tree.nodes], dtype=float)
    root  = np.arange(len(tree.nodes))
    for i, p in enumerate(tree.parent):
        root[i] = i if p < 0 else root[p]
    feats = []
    for i in range(len(tree.nodes)):
        p    = tree.parent[i]
        kids = ch[i]
        feats.append({
            "t_lo":               tree.levels[tree.lo[i]],
            "t_hi":               tree.levels[tree.hi[i]],
            "persistence":        tree.hi[i] - tree.lo[i] + 1,
            "depth":              int(depth[i]),
            "parent_area_ratio":  area[i] / area[p] if p >= 0 else 1.0,
            "root_area_ratio":    area[i] / area[root[i]],
            "n_children":         len(kids),
            "max_child_area_ratio": max((area[c] for c in kids), default=0.0) / area[i],
            "child_area_sum_ratio": sum(area[c] for c in kids) / area[i],
            "is_leaf":            int(not kids),
        })
    return feats


# ── Blob features (26) ────────────────────────────────────────────
RING_PX = 5

FEATURE_NAMES = [
    "log_area", "bbox_h", "bbox_w", "perimeter", "compactness",
    "solidity", "rect_aspect", "pca_ratio", "skeleton_len", "mean_width",
    "n_holes", "p_mean", "p_max", "p_std", "p_q10", "p_q50", "p_q90",
    "ring_p_mean", "int_mean", "int_std", "ring_int_mean", "int_contrast",
    "r_over_R", "img_n_cand", "img_log_total_area", "area_rank_in_img",
]

TREE_FEATURE_NAMES = [
    "t_lo", "t_hi", "persistence", "depth", "parent_area_ratio",
    "root_area_ratio", "n_children", "max_child_area_ratio",
    "child_area_sum_ratio", "is_leaf",
]



def solar_disk(image):
    """Find solar disk center (cx, cy) and radius R."""
    blur    = cv2.GaussianBlur(image, (0, 0), 3)
    _, m    = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, st, cen = cv2.connectedComponentsWithStats(
        (m > 0).astype(np.uint8), connectivity=8)
    if n < 2:
        h, w = image.shape
        return w/2, h/2, 0.44 * min(h, w)
    k      = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    filled = cv2.morphologyEx(
        (lab == k).astype(np.uint8), cv2.MORPH_CLOSE,
        np.ones((31, 31), np.uint8))
    return float(cen[k][0]), float(cen[k][1]), float(np.sqrt(filled.sum() / np.pi))


def extract_instances_at_threshold(prob, threshold=0.5, closing_iterations=2,
                                   connectivity=8, min_pixels=1,
                                   max_instances=0, min_mean_prob=0.0):
    """Extract connected components at a fixed threshold — reference set for image context."""
    mask = prob > threshold
    if closing_iterations:
        mask = ndi.binary_closing(mask, iterations=closing_iterations)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=connectivity)
    nodes = []
    for c in range(1, n):
        x, y, w, h, area = stats[c]
        if area < min_pixels:
            continue
        if min_mean_prob > 0 and prob[y:y+h, x:x+w][labels[y:y+h, x:x+w] == c].mean() < min_mean_prob:
            continue
        crop = labels[y:y+h, x:x+w] == c
        nodes.append(TreeNode(int(y), int(x), crop, int(area)))
    nodes.sort(key=lambda nd: nd.area, reverse=True)
    if max_instances:
        nodes = nodes[:max_instances]
    return nodes


def _shape_features(crop):
    """Shape features for one crop — extracted separately for reuse."""
    c        = crop.astype(np.uint8)
    contours, hierarchy = cv2.findContours(c, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    outer    = [cnt for cnt, h in zip(contours, hierarchy[0]) if h[3] < 0]
    perim    = float(sum(cv2.arcLength(cnt, True) for cnt in outer))
    n_holes  = int(sum(1 for h in hierarchy[0] if h[3] >= 0))
    pts      = np.vstack(outer).reshape(-1, 2).astype(np.float32)
    area     = float(c.sum())
    rw, rh   = cv2.minAreaRect(pts)[1]
    ys, xs   = np.nonzero(c)
    if len(xs) > 2:
        ev        = np.linalg.eigvalsh(np.cov(np.stack([xs, ys]).astype(np.float64)))
        pca_ratio = float(min(np.sqrt(ev[1] / max(ev[0], 1e-6)), 100.0))
    else:
        pca_ratio = 1.0
    skel = int(skeletonize(crop).sum())  # ← skeletonize (not thin, no special case)
    return {
        "perimeter":    perim,
        "compactness":  min(4 * np.pi * area / max(perim, 1)**2, 1.0),
        "solidity":     min(area / max(cv2.contourArea(cv2.convexHull(pts)), 1.0), 1.0),
        "rect_aspect":  max(rw, rh, 1.0) / max(min(rw, rh), 1.0),
        "pca_ratio":    pca_ratio,
        "skeleton_len": float(skel),
        "mean_width":   area / max(skel, 1),
        "n_holes":      n_holes,
    }


def features_for(nodes, prob, image, context_areas=None):
    """Extract 26 blob features for a list of tree nodes.
    
    context_areas: if None uses nodes' own areas for image-level features.
    Pass level-0.5 areas for non-0.5 nodes to match training context.
    """
    cx, cy, R = solar_disk(image)
    kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*RING_PX+1, 2*RING_PX+1))
    H, W      = prob.shape
    areas     = np.array([nd.area for nd in nodes])

    if context_areas is None:
        ctx   = areas
        ranks = (-areas).argsort().argsort() if len(nodes) else areas
    else:
        ctx   = np.asarray(context_areas)
        ranks = np.array([int((ctx > a).sum()) for a in areas])

    n_ctx     = len(ctx)
    log_total = float(np.log(ctx.sum())) if len(ctx) else 0.0
    feats     = []

    for i, node in enumerate(nodes):
        y0 = max(node.y0 - RING_PX, 0)
        x0 = max(node.x0 - RING_PX, 0)
        y1 = min(node.y1 + RING_PX, H)
        x1 = min(node.x1 + RING_PX, W)
        mask = np.zeros((y1-y0, x1-x0), dtype=bool)
        mask[node.y0-y0:node.y1-y0, node.x0-x0:node.x1-x0] = node.crop
        ring    = cv2.dilate(mask.astype(np.uint8), kernel).astype(bool) & ~mask
        p_crop  = prob [y0:y1, x0:x1]
        ip_crop = image[y0:y1, x0:x1].astype(np.float32)
        pv      = p_crop [mask]
        iv      = ip_crop[mask]
        ring_int = float(ip_crop[ring].mean()) if ring.any() else float(iv.mean())
        ys, xs  = np.nonzero(node.crop)
        r = np.hypot(xs.mean() + node.x0 - cx,
                     ys.mean() + node.y0 - cy) / R if R > 0 else 0.0

        sh = _shape_features(node.crop)
        feats.append({
            "log_area":           float(np.log(node.area)),
            "bbox_h":             node.crop.shape[0],
            "bbox_w":             node.crop.shape[1],
            **sh,
            "p_mean":             float(pv.mean()),
            "p_max":              float(pv.max()),
            "p_std":              float(pv.std()),
            "p_q10":              float(np.quantile(pv, 0.1)),
            "p_q50":              float(np.quantile(pv, 0.5)),
            "p_q90":              float(np.quantile(pv, 0.9)),
            "ring_p_mean":        float(p_crop[ring].mean()) if ring.any() else 0.0,
            "int_mean":           float(iv.mean()),
            "int_std":            float(iv.std()),
            "ring_int_mean":      ring_int,
            "int_contrast":       (ring_int - float(iv.mean())) / max(ring_int, 1.0),
            "r_over_R":           float(r),
            "img_n_cand":         n_ctx,
            "img_log_total_area": log_total,
            "area_rank_in_img":   int(ranks[i]),
        })
    return feats


def tree_node_features(prob, image):
    """Build component tree and extract 36 features per node.
    
    Two-pass extraction:
    Pass 1: level-0.5 nodes → use their own areas as image context
    Pass 2: all other nodes → use level-0.5 areas as image context
    This matches the training feature computation exactly.
    """
    import pandas as pd

    tree  = build_tree(prob)
    n     = len(tree)

    # Level-0.5 reference candidates
    cands = extract_instances_at_threshold(prob, **CAND_PARAMS)
    key   = {(tree.nodes[i].y0, tree.nodes[i].x0, tree.nodes[i].area): i
             for i in tree.at_level(0.5)}
    ord05 = [key[(c.y0, c.x0, c.area)] for c in cands
             if (c.y0, c.x0, c.area) in key]

    # Pass 1 — level-0.5 nodes with their own context
    base = [None] * n
    for i, f in zip(ord05, features_for([tree.nodes[i] for i in ord05], prob, image)):
        base[i] = f

    # Pass 2 — all other nodes with level-0.5 areas as context
    others = [i for i in range(n) if base[i] is None]
    ctx    = np.array([tree.nodes[i].area for i in ord05])
    for i, f in zip(others, features_for([tree.nodes[i] for i in others],
                                          prob, image, context_areas=ctx)):
        base[i] = f

    feats = pd.concat([
        pd.DataFrame(base, columns=FEATURE_NAMES, dtype=float),
        pd.DataFrame(tree_features(tree), columns=TREE_FEATURE_NAMES, dtype=float),
    ], axis=1)

    return tree, feats


print("Component tree and feature extraction ready.")
print(f"  Tree levels:         {len(LEVELS)} ({LEVELS[0]}–{LEVELS[-1]})")
print(f"  Blob features:       {len(FEATURE_NAMES)}")
print(f"  Tree features:       {len(TREE_FEATURE_NAMES)}")
print(f"  Total per node:      {len(FEATURE_NAMES) + len(TREE_FEATURE_NAMES)}")

Component tree and feature extraction ready.
  Tree levels:         19 (0.05–0.95)
  Blob features:       26
  Tree features:       10
  Total per node:      36


## 10. OOF Cache *(skippable — prebuilt cache provided)*

Builds the component tree cache from the OOF probability maps (Section 8).
For each of the 707 training images:
- Loads the ensemble OOF probability map (R34 + B4 averaged)
- Builds the component tree and extracts 36 features per node
- Computes IoU between each node and each GT filament instance

**Skip if `cache/tree_ens_r34_b4.pkl` is already available.**

In [ ]:
# ══════════════════════════════════════════════════════════════════
# OPTIONAL — Skip if cache/tree_ens_r34_b4.pkl is available
# Requires OOF prob maps from Section 8
# ══════════════════════════════════════════════════════════════════

import pickle

ENS_DIR    = os.path.join(WORK_DIR, "maps_ens")
CACHE_PATH = os.path.join(WORK_DIR, "cache", "tree_ens_r34_b4.pkl")
os.makedirs(os.path.join(WORK_DIR, "cache"), exist_ok=True)

# Build stem → annotation IDs mapping
stem_to_ids = {}
for img in data["images"]:
    s = Path(img["file_name"]).stem
    stem_to_ids.setdefault(s, []).append(img["id"])

stem_to_img = {Path(img["file_name"]).stem: img for img in data["images"]}

oof_cache = []
npz_paths = sorted(Path(ENS_DIR).glob("*.npz"))
print(f"Building OOF cache for {len(npz_paths)} images...")

for npz_path in tqdm(npz_paths, desc="building cache"):
    stem = npz_path.stem

    # Load ensemble prob map
    with np.load(str(npz_path)) as f:
        prob = f["prob"].astype(np.float32)

    # Load image
    img_info   = stem_to_img[stem]
    image_path = os.path.join(TRAIN_IMAGES_DIR, img_info["file_name"])
    image      = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # Build component tree + extract 36 features
    tree, feats = tree_node_features(prob, image)
    n           = len(tree)

    # GT IoU matrices — one per annotator record
    all_ids = stem_to_ids.get(stem, [])
    mats    = []
    H, W    = image.shape

    for aid in all_ids:
        anns     = [a for a in data["annotations"] if a["image_id"] == aid]
        gt_masks = []
        for ann in anns:
            m = np.zeros((H, W), dtype=np.uint8)
            for poly in ann["segmentation"]:
                pts = np.array(poly, dtype=np.int32).reshape(-1, 2)
                cv2.fillPoly(m, [pts], 1)
            n2, lab2, _, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
            for k in range(1, n2):
                gt_masks.append((lab2 == k).astype(bool))

        # IoU matrix: (n_gt, n_nodes)
        iou_mat = np.zeros((len(gt_masks), n), dtype=np.float32)
        for gi, gt in enumerate(gt_masks):
            gt_rows = np.flatnonzero(gt.any(axis=1))
            gt_cols = np.flatnonzero(gt.any(axis=0))
            if len(gt_rows) == 0:
                continue
            gy0, gy1 = int(gt_rows[0]), int(gt_rows[-1]) + 1
            gx0, gx1 = int(gt_cols[0]), int(gt_cols[-1]) + 1
            gt_area  = int(gt.sum())
            for ni, nd in enumerate(tree.nodes):
                y0 = max(nd.y0, gy0); y1 = min(nd.y1, gy1)
                x0 = max(nd.x0, gx0); x1 = min(nd.x1, gx1)
                if y0 >= y1 or x0 >= x1:
                    continue
                inter = int((nd.crop[y0-nd.y0:y1-nd.y0, x0-nd.x0:x1-nd.x0] &
                             gt[y0:y1, x0:x1]).sum())
                if inter == 0:
                    continue
                union = nd.area + gt_area - inter
                iou_mat[gi, ni] = inter / union if union > 0 else 0.0
        mats.append(iou_mat)

    oof_cache.append({
        "stem":   stem,
        "R":      len(all_ids),
        "feats":  feats,
        "area":   np.array([nd.area for nd in tree.nodes], dtype=int),
        "parent": np.array(tree.parent, dtype=int),
        "lo":     np.array(tree.lo, dtype=int),
        "hi":     np.array(tree.hi, dtype=int),
        "mats":   mats,
        "nodes":  tree.nodes,
    })

with open(CACHE_PATH, "wb") as f:
    pickle.dump(oof_cache, f)

print(f"OOF cache saved → {CACHE_PATH}")
print(f"  {len(oof_cache)} images")

## 11. Load OOF Cache

Loads the prebuilt component tree cache for the 707 OOF training images.
Each entry contains the component tree, 36 node features, and GT IoU matrices
used to train the LightGBM selector in Section 12.

In [12]:
import pickle

CACHE_PATH = os.path.join(WORK_DIR, "cache", "tree_ens_r34_b4.pkl")

print("Loading OOF cache...")
with open(CACHE_PATH, "rb") as f:
    _cache = pickle.load(f)

oof_imgs = _cache['data']

print(f"  {len(oof_imgs)} images loaded")
print(f"  Example: {oof_imgs[0]['stem']}")
print(f"  Nodes in first image: {len(oof_imgs[0]['feats'])}")
print(f"  Features per node: {oof_imgs[0]['feats'].shape[1]}")
print(f"  Annotators in first image: {oof_imgs[0]['R']}")

Loading OOF cache...
  707 images loaded
  Example: 20110109104734Ch
  Nodes in first image: 118
  Features per node: 36
  Annotators in first image: 1


## 12. Selector

The candidate selector uses a blend of three LightGBM models:
- **Hurdle classifier**: P(node matches a GT filament, IoU > 0.5)
- **Hurdle regressor**: E[IoU | match], clipped to [0.5, 1.0]
- **LambdaRank**: ranking score within each tree root

Final score per node: `h + α × (r − max_r_in_root) / sd`
where `h = P(match) × clip(E[IoU|match], 0.5, 1)`, `α = 0.02`, `τ = 0.24`

A pretrained selector is provided — training (Section 12.3) is skippable.

### 12.1 BlendSelector

In [13]:
import lightgbm as lgb
import json
import hashlib

# ── BlendSelector ─────────────────────────────────────────────────
# Hurdle (classifier × regressor) + alpha × LambdaRank blend
# Applied per component tree; score averaged over RF-DETR models

IOU_CLIP = (0.5, 1.0)

def _tree_roots(parent):
    parent = np.asarray(parent, dtype=int)
    r = np.arange(len(parent))
    for i, p in enumerate(parent):
        if p >= 0:
            r[i] = r[p]
    return r

def _root_max(root, r):
    mx = np.full(len(r), -np.inf)
    np.maximum.at(mx, root, r)
    return mx[root]

def _blend(h, r, root, alpha, sd):
    return h + alpha * (r - _root_max(root, r)) / sd if len(r) else h


class BlendSelector:
    def __init__(self, clf, reg, rank, columns, alpha, tau, sd, meta=None):
        self.clf, self.reg, self.rank = clf, reg, rank
        self.columns = list(columns)
        self.alpha   = float(alpha)
        self.tau     = float(tau)
        self.sd      = float(sd)
        self.meta    = dict(meta or {})

    @property
    def det_min_score(self):
        return float(self.meta.get("det_min_score", 0.05))

    def _X(self, feats):
        missing = [c for c in self.columns if c not in feats.columns]
        if missing:
            raise KeyError(f"missing columns: {missing}")
        return feats[self.columns]

    def hurdle(self, feats):
        X = self._X(feats)
        return self.clf.predict(X) * np.clip(self.reg.predict(X), *IOU_CLIP)

    def rank_score(self, feats):
        return self.rank.predict(self._X(feats))

    def score(self, feats, parent):
        if len(feats) == 0:
            return np.zeros(0)
        return _blend(self.hurdle(feats), self.rank_score(feats),
                      _tree_roots(parent), self.alpha, self.sd)

    def score_mean(self, feats_per_model, parent):
        if len(parent) == 0:
            return np.zeros(0)
        return np.mean([self.score(f, parent) for f in feats_per_model], axis=0)

    @classmethod
    def load(cls, model_dir, verify=True):
        d   = Path(model_dir)
        doc = json.loads((d / "selector.json").read_text(encoding="utf-8"))
        if verify:
            for name, h in doc["sha256"].items():
                actual = hashlib.sha256((d / name).read_bytes()).hexdigest()
                if actual != h:
                    raise ValueError(f"{name}: sha256 mismatch")
        boosters = {k: lgb.Booster(model_file=str(d / doc["files"][k]))
                    for k in ("clf", "reg", "rank")}
        meta = {k: v for k, v in doc.items()
                if k not in ("columns", "alpha", "tau", "sd")}
        return cls(boosters["clf"], boosters["reg"], boosters["rank"],
                   doc["columns"], doc["alpha"], doc["tau"], doc["sd"], meta)

print("BlendSelector ready.")

BlendSelector ready.


### 12.2 Antichain Selection (DP)

In [14]:
# ── DP antichain selection ────────────────────────────────────────
# Selects non-nested nodes maximising sum(score - tau)
# Parents always precede children in tree order

def dp_select(parent, score, allowed):
    n         = len(parent)
    best      = np.zeros(n)
    take      = np.zeros(n, dtype=bool)
    child_sum = np.zeros(n)
    for i in range(n - 1, -1, -1):
        own = score[i] if allowed[i] else -np.inf
        if own > child_sum[i] and own > 0:
            best[i], take[i] = own, True
        else:
            best[i] = max(child_sum[i], 0.0)
        if parent[i] >= 0:
            child_sum[parent[i]] += best[i]
    sel, stack = [], [i for i in range(n) if parent[i] < 0]
    while stack:
        i = stack.pop()
        if take[i]:
            sel.append(i)
        elif best[i] > 0:
            stack.extend(np.flatnonzero(np.array(parent) == i).tolist())
    return np.array(sorted(sel), dtype=int)


def select_tree(parent, score, tau):
    parent = np.asarray(parent, dtype=int)
    return dp_select(
        parent,
        np.where(np.isfinite(score), score - tau, -np.inf),
        np.ones(len(parent), dtype=bool)
    )

print("DP antichain selection ready.")

DP antichain selection ready.


### 12.3 Train Selector *(skippable — pretrained selector provided)*

Trains three LightGBM models on the 707 OOF component trees:
1. **Hurdle classifier**: `P(node matches GT filament, IoU > 0.5)`
2. **Hurdle regressor**: `E[IoU | match]`
3. **LambdaRank ranker**: ranking score within each tree root

Cross-fits `α` and `τ` over 5 folds. Final models fitted on all 707 images.
Requires `cache/tree_ens_r34_b4.pkl` and `cache/rfdetr_feats_fold{0-4}.pkl`.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# OPTIONAL — Skip if using pretrained selector (Section 12.4)
# Requires cache/tree_ens_r34_b4.pkl and cache/rfdetr_feats_fold{0-4}.pkl
# ══════════════════════════════════════════════════════════════════

import pickle
from lightgbm import LGBMClassifier, LGBMRegressor, LGBMRanker
from sklearn.model_selection import StratifiedKFold

CACHE_PATH   = os.path.join(WORK_DIR, "cache", "tree_ens_r34_b4.pkl")
RFDETR_CACHE = os.path.join(WORK_DIR, "cache", "rfdetr_feats_fold{}.pkl")
SELECTOR_DIR = os.path.join(WORK_DIR, "models", "selector")
DET_COLS     = ["det_iou", "det_score", "det_cov", "det_n"]

ALL_FEATURE_NAMES = FEATURE_NAMES + TREE_FEATURE_NAMES

LGBM_PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=7,
                   min_child_samples=20, subsample=0.8, subsample_freq=1,
                   colsample_bytree=0.8, reg_lambda=1.0, random_state=0,
                   verbose=-1, n_jobs=4)

ALPHAS   = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
TAU_GRID = np.round(np.arange(0.02, 0.981, 0.01), 2)

# ── Load RF-DETR OOF features ─────────────────────────────────────
print("Loading RF-DETR OOF features...")
rfdetr_feats = {}
for fold_idx in range(N_FOLDS):
    with open(RFDETR_CACHE.format(fold_idx), "rb") as f:
        rfdetr_feats[fold_idx] = pickle.load(f)
print(f"  Loaded {N_FOLDS} fold RF-DETR feature files")

# ── Merge RF-DETR features into OOF cache ────────────────────────
for im in oof_imgs:
    stem = im["stem"]
    for fold_idx in range(N_FOLDS):
        if stem in rfdetr_feats[fold_idx]:
            det = rfdetr_feats[fold_idx][stem]
            for col in DET_COLS:
                im["feats"][col] = det[col].values
            break

# ── Tree roots ────────────────────────────────────────────────────
def _roots(parent):
    r = np.arange(len(parent))
    for i, p in enumerate(parent):
        if p >= 0:
            r[i] = r[p]
    return r

for im in oof_imgs:
    im["root"] = _roots(im["parent"])

# ── Hurdle fit + predict ──────────────────────────────────────────
def _hurdle_fit_predict(train, test, seed):
    cols = ALL_FEATURE_NAMES + DET_COLS
    X_tr, y_clf, y_reg = [], [], []
    for im in train:
        if len(im["feats"]) == 0:
            continue
        X  = im["feats"][cols].values
        mi = np.array(im["mats"])
        for r in range(len(mi)):
            matched_iou = np.where(mi[r] > 0.5, mi[r], 0).max(axis=0) \
                          if mi[r].shape[0] else np.zeros(len(X))
            y_clf.append((matched_iou > 0).astype(float))
            y_reg.append(matched_iou)
            X_tr.append(X)
    X_tr  = np.vstack(X_tr)
    y_clf = np.concatenate(y_clf)
    y_reg = np.concatenate(y_reg)
    pos   = y_clf > 0
    clf   = LGBMClassifier(**LGBM_PARAMS).fit(X_tr, y_clf)
    reg   = LGBMRegressor(**LGBM_PARAMS).fit(X_tr[pos], y_reg[pos])
    preds = []
    for im in test:
        if len(im["feats"]) == 0:
            preds.append(np.zeros(0))
            continue
        X = im["feats"][cols].values
        h = clf.predict_proba(X)[:, 1] * np.clip(reg.predict(X), *IOU_CLIP)
        preds.append(h)
    return preds, clf, reg


# ── LambdaRank fit + predict ──────────────────────────────────────
def _rank_fit_predict(train, test, seed):
    cols = ALL_FEATURE_NAMES + DET_COLS
    X_tr, y_tr, groups = [], [], []
    for im in train:
        if len(im["feats"]) == 0:
            continue
        order  = np.argsort(im["root"], kind="stable")
        X_tr.append(im["feats"][cols].iloc[order].values)
        u_star = np.array(im["mats"]).mean(axis=0).clip(0) \
                 if len(im["mats"]) else np.zeros(len(im["feats"]))
        u_star = np.where(u_star > 0.5, u_star, 0)
        y_tr.append(np.rint(10 * u_star[order]).astype(int))
        groups.append(np.unique(im["root"][order], return_counts=True)[1])
    ranker = LGBMRanker(objective="lambdarank", **LGBM_PARAMS)
    ranker.fit(np.vstack(X_tr), np.concatenate(y_tr),
               group=np.concatenate(groups))
    preds = []
    for im in test:
        if len(im["feats"]) == 0:
            preds.append(np.zeros(0))
            continue
        preds.append(ranker.predict(im["feats"][cols].values))
    return preds, ranker


# ── PQ from scores using stored IoU matrices ──────────────────────
def _pq_from_scores(imgs, scores, tau):
    total_pq = []
    for im, sc in zip(imgs, scores):
        parent = np.asarray(im["parent"], dtype=int)
        sel    = select_tree(parent, sc, tau)
        if len(sel) == 0:
            total_pq.append(0.0)
            continue
        iou_mat = np.array(im["mats"])  # (R, n_nodes)
        pqs = []
        for r in range(iou_mat.shape[0]):
            row      = iou_mat[r]
            iou_sel  = row[:, sel] if row.shape[0] else np.zeros((0, len(sel)))
            matched  = iou_sel.max(axis=1) > 0.5 if iou_sel.shape[0] else np.zeros(0, bool)
            tp       = int(matched.sum())
            fp       = len(sel) - tp
            fn       = row.shape[0] - tp
            sq       = float(iou_sel.max(axis=1)[matched].mean()) if tp > 0 else 0.0
            rq       = tp / (tp + 0.5*fp + 0.5*fn) if (tp+fp+fn) > 0 else 0.0
            pqs.append(sq * rq)
        total_pq.append(float(np.mean(pqs)) if pqs else 0.0)
    return float(np.mean(total_pq))


# ── Cross-fit alpha + tau ─────────────────────────────────────────
print("Cross-fitting alpha and tau...")
R    = np.array([im["R"] for im in oof_imgs])
h_cv = [None] * len(oof_imgs)
r_cv = [None] * len(oof_imgs)

for k, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(
        np.zeros(len(oof_imgs)), R)):
    train_imgs = [oof_imgs[i] for i in tr]
    test_imgs  = [oof_imgs[i] for i in te]
    h_te, _, _ = _hurdle_fit_predict(train_imgs, test_imgs, seed=0)
    r_te, _    = _rank_fit_predict(train_imgs, test_imgs, seed=0)
    for i, h, r in zip(te, h_te, r_te):
        h_cv[i] = h
        r_cv[i] = r
    print(f"  Fold {k+1}/5 done", flush=True)

sd = float(np.std(np.concatenate([r for r in r_cv if len(r)])))
print(f"  sd={sd:.4f}")

# Grid search over alpha × tau
best_pq, best_alpha, best_tau_val = -1, 0.02, 0.24
grid_results = []
for alpha in ALPHAS:
    scores = [_blend(h, r, _tree_roots(im["parent"]), alpha, sd)
              for im, h, r in zip(oof_imgs, h_cv, r_cv)]
    for tau in TAU_GRID:
        pq = _pq_from_scores(oof_imgs, scores, tau)
        grid_results.append({"alpha": alpha, "tau": float(tau), "cv_pq": pq})
        if pq > best_pq or (pq == best_pq and alpha < best_alpha):
            best_pq, best_alpha, best_tau_val = pq, alpha, float(tau)

print(f"Best: alpha={best_alpha}, tau={best_tau_val}, cv_pq={best_pq:.5f}")

# ── Final fit on all 707 images ───────────────────────────────────
print("Fitting final models on all 707 images...")
_, final_clf, final_reg = _hurdle_fit_predict(oof_imgs, [], seed=0)
_, final_ranker          = _rank_fit_predict(oof_imgs, [], seed=0)

# ── Save trained selector ─────────────────────────────────────────
os.makedirs(SELECTOR_DIR, exist_ok=True)
final_clf.booster_.save_model(os.path.join(SELECTOR_DIR, "hurdle_clf.txt"))
final_reg.booster_.save_model(os.path.join(SELECTOR_DIR, "hurdle_reg.txt"))
final_ranker.booster_.save_model(os.path.join(SELECTOR_DIR, "ranker.txt"))

selector_meta = {
    "format": "filament-blend-selector/1",
    "columns": ALL_FEATURE_NAMES + DET_COLS,
    "alpha": best_alpha, "tau": best_tau_val, "sd": sd,
    "det_min_score": 0.3,
    "files": {"clf": "hurdle_clf.txt", "reg": "hurdle_reg.txt", "rank": "ranker.txt"},
    "sha256": {
        "hurdle_clf.txt": hashlib.sha256(
            open(os.path.join(SELECTOR_DIR, "hurdle_clf.txt"), "rb").read()).hexdigest(),
        "hurdle_reg.txt": hashlib.sha256(
            open(os.path.join(SELECTOR_DIR, "hurdle_reg.txt"), "rb").read()).hexdigest(),
        "ranker.txt": hashlib.sha256(
            open(os.path.join(SELECTOR_DIR, "ranker.txt"), "rb").read()).hexdigest(),
    }
}
with open(os.path.join(SELECTOR_DIR, "selector.json"), "w") as f:
    json.dump(selector_meta, f, indent=2)

print(f"Selector saved to {SELECTOR_DIR}")
print(f"  alpha={best_alpha}, tau={best_tau_val}, sd={sd:.4f}")

### 12.4 Load Pretrained Selector

Loads the pretrained selector directly — `α = 0.02`, `τ = 0.24`, `sd = 2.135`, OOF PQ = 0.4665

In [15]:
SELECTOR_DIR = os.path.join(WORK_DIR, "models", "selector")

selector = BlendSelector.load(SELECTOR_DIR)
print(f"Selector loaded: {len(selector.columns)} columns")
print(f"  alpha={selector.alpha} | tau={selector.tau} | sd={selector.sd:.4f}")
print(f"  det_min_score={selector.det_min_score}")
print(f"  det columns: {[c for c in selector.columns if c.startswith('det_')]}")

Selector loaded: 40 columns
  alpha=0.02 | tau=0.24 | sd=2.1351
  det_min_score=0.3
  det columns: ['det_iou', 'det_score', 'det_cov', 'det_n']


## 13. Test Probability Maps

Runs D4 TTA inference on all 180 test images using all 10 trained models
(R34 × 5 folds + B4 × 5 folds). All models predict all test images —
no held-out constraint applies to test data.

The 10 probability maps per image are averaged to form the ensemble map
used for component tree construction in Section 15.

In [17]:
TEST_IMAGES_DIR = os.path.join(DATA_DIR, "test", "test_images")
TEST_MAPS_DIR   = os.path.join(WORK_DIR, "test_maps")
os.makedirs(TEST_MAPS_DIR, exist_ok=True)

# Get all test image stems
test_stems = sorted([Path(p).stem for p in Path(TEST_IMAGES_DIR).glob("*.jpeg")])
print(f"Test images: {len(test_stems)}")

# ── Generate test prob maps ───────────────────────────────────────
for suffix, models in (("r34", r34_models), ("b4", b4_models)):
    for fold_idx, model in enumerate(models):
        out_dir   = os.path.join(TEST_MAPS_DIR, f"{suffix}_fold{fold_idx}")
        done_file = os.path.join(out_dir, "DONE")
        os.makedirs(out_dir, exist_ok=True)
        if os.path.exists(done_file):
            print(f"  {suffix} fold {fold_idx} already done — skipping")
            continue
        print(f"\n{suffix} fold {fold_idx} — {len(test_stems)} test images")
        t0 = time.time()
        for stem in tqdm(test_stems, desc=f"{suffix}_fold{fold_idx}"):
            out_path = os.path.join(out_dir, f"{stem}.npz")
            if os.path.exists(out_path):
                continue
            image_path = os.path.join(TEST_IMAGES_DIR, f"{stem}.jpeg")
            prob       = predict_full_image_d4tta(model, image_path, device)
            save_prob(out_path, prob)
        elapsed = (time.time() - t0) / 60
        open(done_file, "w").write(f"{elapsed:.1f} min")
        print(f"  done in {elapsed:.1f} min")

# ── Ensemble test prob maps ───────────────────────────────────────
TEST_ENS_DIR = os.path.join(WORK_DIR, "test_maps_ens")
os.makedirs(TEST_ENS_DIR, exist_ok=True)

print("\nBuilding ensemble test prob maps...")
for stem in tqdm(test_stems, desc="ensemble"):
    out_path = os.path.join(TEST_ENS_DIR, f"{stem}.npz")
    if os.path.exists(out_path):
        continue
    maps = []
    for suffix in ("r34", "b4"):
        for fold_idx in range(N_FOLDS):
            npz = os.path.join(TEST_MAPS_DIR, f"{suffix}_fold{fold_idx}", f"{stem}.npz")
            if os.path.exists(npz):
                maps.append(load_prob(npz))
    if maps:
        save_prob(out_path, np.mean(maps, axis=0))

print(f"Test ensemble maps saved → {TEST_ENS_DIR}")
print(f"  {len(list(Path(TEST_ENS_DIR).glob('*.npz')))} images")

Test images: 180
  r34 fold 0 already done — skipping
  r34 fold 1 already done — skipping
  r34 fold 2 already done — skipping
  r34 fold 3 already done — skipping
  r34 fold 4 already done — skipping
  b4 fold 0 already done — skipping
  b4 fold 1 already done — skipping
  b4 fold 2 already done — skipping
  b4 fold 3 already done — skipping
  b4 fold 4 already done — skipping

Building ensemble test prob maps...


ensemble: 100%|██████████| 180/180 [00:00<00:00, 157581.87it/s]

Test ensemble maps saved → /home/seif/solar-filament-segmentation/test_maps_ens
  180 images


## 14. RF-DETR Inference

Runs RF-DETR-Seg-Large on all 180 test images using all 5 trained fold models.
Each model produces instance masks with confidence scores (threshold 0.05).
The 5 sets of instances are used separately in Section 15 — one det-column set
per RF-DETR model, scores averaged over models by the selector.

In [18]:
from PIL import Image
from pycocotools import mask as mask_utils

RFDETR_INSTANCES_DIR = os.path.join(WORK_DIR, "rfdetr_instances")
os.makedirs(RFDETR_INSTANCES_DIR, exist_ok=True)

RFDETR_THRESHOLD = 0.05  # keep all instances with score >= 0.05

def encode_rle(mask):
    """Encode boolean mask as COCO RLE string."""
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("utf-8") if isinstance(rle["counts"], bytes) else rle["counts"]

def decode_rle(counts, shape=(2048, 2048)):
    """Decode COCO RLE string to boolean mask."""
    return mask_utils.decode({"size": list(shape), "counts": counts}).astype(bool)

# ── RF-DETR inference per fold ────────────────────────────────────
# Each fold model predicts all 180 test images
# Instances saved as JSON lines (one line per image)

for fold_idx, model in enumerate(rfdetr_models):
    out_path  = os.path.join(RFDETR_INSTANCES_DIR, f"instances_fold{fold_idx}.jsonl")
    done_file = out_path + ".DONE"
    if os.path.exists(done_file):
        print(f"  fold {fold_idx} already done — skipping")
        continue

    # Load already-done stems
    done_stems = set()
    if os.path.exists(out_path):
        for ln in open(out_path).readlines():
            done_stems.add(json.loads(ln)["stem"])

    print(f"\nRF-DETR fold {fold_idx} — {len(test_stems)} test images")
    t0 = time.time()

    with open(out_path, "a") as fh:
        for stem in tqdm(test_stems, desc=f"rfdetr_fold{fold_idx}"):
            if stem in done_stems:
                continue
            image_path = os.path.join(TEST_IMAGES_DIR, f"{stem}.jpeg")
            img        = Image.open(image_path).convert("RGB")
            det        = model.predict(img, threshold=RFDETR_THRESHOLD)
            instances  = []
            masks      = det.mask if det.mask is not None else np.zeros((0, 2048, 2048), bool)
            for m, sc in zip(masks, det.confidence):
                m = np.asarray(m, bool)
                if m.shape != (2048, 2048):
                    m = np.asarray(
                        Image.fromarray(m.astype(np.uint8) * 255).resize(
                            (2048, 2048), Image.NEAREST)) > 127
                instances.append({"rle": encode_rle(m), "score": float(sc)})
            fh.write(json.dumps({"stem": stem, "fold": fold_idx,
                                 "instances": instances}) + "\n")
            fh.flush()

    elapsed = (time.time() - t0) / 60
    open(done_file, "w").write(f"{elapsed:.1f} min")
    print(f"  fold {fold_idx} done in {elapsed:.1f} min — {len(test_stems)} images")

print(f"\nRF-DETR instances saved → {RFDETR_INSTANCES_DIR}")
for fold_idx in range(N_FOLDS):
    path = os.path.join(RFDETR_INSTANCES_DIR, f"instances_fold{fold_idx}.jsonl")
    n    = sum(1 for _ in open(path)) if os.path.exists(path) else 0
    print(f"  fold {fold_idx}: {n} images")

  fold 0 already done — skipping
  fold 1 already done — skipping
  fold 2 already done — skipping
  fold 3 already done — skipping
  fold 4 already done — skipping

RF-DETR instances saved → /home/seif/solar-filament-segmentation/rfdetr_instances
  fold 0: 180 images
  fold 1: 180 images
  fold 2: 180 images
  fold 3: 180 images
  fold 4: 180 images


## 15. Generate Submission

Applies the full pipeline to all 180 test images and writes `submission.csv`.

In [21]:
import pandas as pd
from pycocotools import mask as mask_utils

SUBMISSION_PATH = os.path.join(WORK_DIR, "submission.csv")
DET_FEATURE_NAMES = ["det_iou", "det_score", "det_cov", "det_n"]
DET_MIN_SCORE     = selector.det_min_score  # 0.3
CONF              = 0.3
IMAGE_SIZE        = (2048, 2048)

# ── RLE encoding ──────────────────────────────────────────────────
def instance_to_mask(node, shape=IMAGE_SIZE):
    """Convert tree node to full-size boolean mask."""
    m = np.zeros(shape, dtype=bool)
    m[node.y0:node.y1, node.x0:node.x1] = node.crop
    return m

def mask_to_rle(mask):
    """Encode boolean mask as COCO RLE string."""
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("utf-8") if isinstance(rle["counts"], bytes) else rle["counts"]

# ── RF-DETR node features ─────────────────────────────────────────
def inter_bbox(a_node, b_mask_crop, b_y0, b_x0):
    """Intersection of tree node and instance crop."""
    y0 = max(a_node.y0, b_y0)
    y1 = min(a_node.y1, b_y0 + b_mask_crop.shape[0])
    x0 = max(a_node.x0, b_x0)
    x1 = min(a_node.x1, b_x0 + b_mask_crop.shape[1])
    if y0 >= y1 or x0 >= x1:
        return 0
    return int((a_node.crop[y0-a_node.y0:y1-a_node.y0, x0-a_node.x0:x1-a_node.x0] &
                b_mask_crop[y0-b_y0:y1-b_y0, x0-b_x0:x1-b_x0]).sum())


def compute_det_features(tree, instances, size=2048):
    """Compute 4 RF-DETR det columns for all tree nodes.
    
    instances: list of (mask_bool_2048x2048, score)
    Returns DataFrame with det_iou, det_score, det_cov, det_n
    """
    n  = len(tree.nodes)
    f  = np.zeros((n, 4), dtype=np.float32)

    # Build confidence coverage map (score >= CONF)
    conf_map = np.zeros(IMAGE_SIZE, dtype=bool)
    inst_decoded = []
    for mask, score in instances:
        # Get tight bbox
        rows = np.flatnonzero(mask.any(axis=1))
        cols = np.flatnonzero(mask.any(axis=0))
        if len(rows) == 0:
            inst_decoded.append(None)
            continue
        y0, y1 = int(rows[0]), int(rows[-1]) + 1
        x0, x1 = int(cols[0]), int(cols[-1]) + 1
        crop    = mask[y0:y1, x0:x1]
        inst_decoded.append((crop, y0, x0, int(mask.sum()), score))
        if score >= CONF:
            conf_map[y0:y1, x0:x1] |= crop

    for j, nd in enumerate(tree.nodes):
        best_iou   = 0.0
        best_score = 0.0
        n_conf     = 0
        for inst in inst_decoded:
            if inst is None:
                continue
            crop, iy0, ix0, iarea, iscore = inst
            i = inter_bbox(nd, crop, iy0, ix0)
            if i == 0:
                continue
            if iscore >= CONF:
                n_conf += 1
            iou = i / (nd.area + iarea - i)
            if iou > best_iou:
                best_iou   = iou
                best_score = iscore
        # det_cov: fraction of node covered by confident instances
        cov = (conf_map[nd.y0:nd.y1, nd.x0:nd.x1] & nd.crop).sum() / nd.area
        f[j] = [best_iou, best_score, float(cov), float(n_conf)]

    return pd.DataFrame(f, columns=DET_FEATURE_NAMES)


# ── Load RF-DETR instances ────────────────────────────────────────
def load_rfdetr_instances(fold_idx, stem):
    """Load RF-DETR instances for one image from one fold."""
    path = os.path.join(RFDETR_INSTANCES_DIR, f"instances_fold{fold_idx}.jsonl")
    for ln in open(path):
        r = json.loads(ln)
        if r["stem"] == stem:
            instances = []
            for inst in r["instances"]:
                if inst["score"] < DET_MIN_SCORE:
                    continue
                mask = decode_rle(inst["rle"])
                instances.append((mask, inst["score"]))
            return instances
    return []


# ── Generate submission ───────────────────────────────────────────
print("Generating submission...")
rows = []

for stem in tqdm(test_stems, desc="submission"):
    # Load ensemble prob map
    ens_path = os.path.join(TEST_ENS_DIR, f"{stem}.npz")
    prob     = load_prob(ens_path)

    # Load image
    image_path = os.path.join(TEST_IMAGES_DIR, f"{stem}.jpeg")
    image      = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # Build component tree + extract 36 features
    tree, feats = tree_node_features(prob, image)

    if len(tree) == 0:
        continue

    parent = np.asarray(tree.parent, dtype=int)

    # Compute det columns per RF-DETR fold model
    feats_per_model = []
    for fold_idx in range(N_FOLDS):
        instances = load_rfdetr_instances(fold_idx, stem)
        det_feats = compute_det_features(tree, instances)
        feats_per_model.append(
            pd.concat([feats.reset_index(drop=True),
                       det_feats.reset_index(drop=True)], axis=1)
        )

    # Score nodes — averaged over RF-DETR models
    score = selector.score_mean(feats_per_model, parent)

    # DP antichain selection
    sel   = select_tree(parent, score, selector.tau)

    # Encode selected nodes as RLE
    nodes = sorted((tree.nodes[i] for i in sel),
                   key=lambda nd: nd.area, reverse=True)
    for k, nd in enumerate(nodes, start=1):
        mask = instance_to_mask(nd)
        rows.append({
            "filament_id":     f"{stem}_{k}",
            "segmentation_rle": mask_to_rle(mask),
        })

# Save submission
sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSubmission saved → {SUBMISSION_PATH}")
print(f"  {len(sub)} filaments across {sub['filament_id'].str.split('_').str[0].nunique()} images")

Generating submission...


submission: 100%|██████████| 180/180 [11:35<00:00,  3.86s/it]


Submission saved → /home/seif/solar-filament-segmentation/submission.csv
  1165 filaments across 177 images
